In [1]:
import numpy as np
import pandas as pd
import polars as pl
from polars import selectors as cs
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from pathlib import Path
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OrdinalEncoder, OneHotEncoder, TargetEncoder, Binarizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import train_test_split,cross_validate,KFold,GridSearchCV,RandomizedSearchCV
from sklearn.metrics import root_mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.feature_selection import SelectFromModel
from sklearn.utils.validation import check_is_fitted
from category_encoders import CountEncoder

%matplotlib inline

from add_features import add_modified_features
warnings.filterwarnings('ignore')

## Data Preprocessing

In [ ]:
def load_data():
    # Read
    dir = Path("../../data/")
    train = pd.read_csv(dir / "train.csv", index_col='Id')
    test = pd.read_csv(dir / "test.csv", index_col='Id')
    # filter outlier
    outlier_ids = [524,1299]  # 確認した外れ値のId
    train = train.filter(~pl.col('Id').is_in(outlier_ids))
    # Preprocessing
    df= pd.concat([train, test])
    df = clean(df)
    df = impute(df)
    # Reform splits
    df_train = df.loc[df_train.index, :]
    df_test = df.loc[df_test.index, :]
    return df_train, df_test


### Clean Data

In [12]:
df = pl.read_csv('../../data/train.csv',infer_schema_length=None,null_values='NA')
df_strings = df.select(cs.string()).to_pandas()
# {列: {値: 出現回数}}の辞書を作成
value_counts_dict = {
    col: df_strings[col].value_counts().to_dict() for col in df_strings.columns}
value_counts_dict

{'MSZoning': {'RL': 1151, 'RM': 218, 'FV': 65, 'RH': 16, 'C (all)': 10},
 'Street': {'Pave': 1454, 'Grvl': 6},
 'Alley': {'Grvl': 50, 'Pave': 41},
 'LotShape': {'Reg': 925, 'IR1': 484, 'IR2': 41, 'IR3': 10},
 'LandContour': {'Lvl': 1311, 'Bnk': 63, 'HLS': 50, 'Low': 36},
 'Utilities': {'AllPub': 1459, 'NoSeWa': 1},
 'LotConfig': {'Inside': 1052,
  'Corner': 263,
  'CulDSac': 94,
  'FR2': 47,
  'FR3': 4},
 'LandSlope': {'Gtl': 1382, 'Mod': 65, 'Sev': 13},
 'Neighborhood': {'NAmes': 225,
  'CollgCr': 150,
  'OldTown': 113,
  'Edwards': 100,
  'Somerst': 86,
  'Gilbert': 79,
  'NridgHt': 77,
  'Sawyer': 74,
  'NWAmes': 73,
  'SawyerW': 59,
  'BrkSide': 58,
  'Crawfor': 51,
  'Mitchel': 49,
  'NoRidge': 41,
  'Timber': 38,
  'IDOTRR': 37,
  'ClearCr': 28,
  'StoneBr': 25,
  'SWISU': 25,
  'MeadowV': 17,
  'Blmngtn': 17,
  'BrDale': 16,
  'Veenker': 11,
  'NPkVill': 9,
  'Blueste': 2},
 'Condition1': {'Norm': 1260,
  'Feedr': 81,
  'Artery': 48,
  'RRAn': 26,
  'PosN': 19,
  'RRAe': 11,
  '

In [17]:
df['GarageYrBlt'].describe()

statistic,value
str,f64
"""count""",1379.0
"""null_count""",81.0
"""mean""",1978.506164
"""std""",24.689725
"""min""",1900.0
"""25%""",1961.0
"""50%""",1980.0
"""75%""",2002.0
"""max""",2010.0


In [ ]:
def clean(df:pd.DataFrame):
    

### pipeline

In [ ]:
from notebooks.baseline_v1.preprocess import build_target_transformer, build_preprocessor

log_standardize_y = build_target_transformer()
preprocessor = build_preprocessor()

### add modified -> drop useless -> filter outlier ->train test split

In [ ]:
# add modified
X = add_modified_features(X)
X_test = add_modified_features(X_test)

# drop useless
drop_feats = [
    'Id',
    '2ndFlrSF',
    '1stFlrSF',
    'GrLivArea',
    'Exterior1st',
    'Exterior2nd',
    'TotalBsmtSF',
    'GarageCars',
    'TotalFlrSF',
    'YearBuilt',
    'OverallQual',
    'BsmtFullBath',
]
X = X.select(cs.exclude(drop_feats))
X_test = X_test.select(cs.exclude(drop_feats))

# split
X_train, X_va, y_train, y_va = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
feat_corr = X_train.select(cs.numeric()).to_pandas().corr()

corr_threshold = 0.8

# 上三角だけを使って、同じ組み合わせの重複と自己相関(対角)を除外
corr_pairs = (
    feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_1", "feature_2", "corr"]

high_corr_pairs = (
    corr_pairs.loc[corr_pairs["corr"].abs() > corr_threshold]
    .sort_values("corr", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

high_corr_pairs

### Ridge

In [ ]:
N_SELECT_FEATS = 30

base_model = Pipeline([
    ("prep", preprocessor),
    # ("select", SelectFromModel(Ridge(alpha=1.0), max_features=N_SELECT_FEATS, threshold=-np.inf)),
    ("ridge", Ridge(alpha=1.0))
])
model = TransformedTargetRegressor(
    regressor=base_model,
    transformer=log_standardize_y,
    check_inverse=False
)

folds = 5
cv = KFold(n_splits=folds, shuffle=True, random_state=42)
result = cross_validate(
    model, X_train, y_train,
    cv=cv,
    scoring="neg_root_mean_squared_log_error",
    return_train_score=True,
)

# スコア表示
train_scores = -result["train_score"]
valid_scores = -result["test_score"]

for i, (tr, va) in enumerate(zip(train_scores, valid_scores), 1):
    print(f"Fold {i}: train RMSLE = {tr:.4f} / valid RMSLE = {va:.4f}")
print(f"CV平均: train {train_scores.mean():.4f} / valid {valid_scores.mean():.4f}")

model.fit(X_train, y_train)
test_score = root_mean_squared_log_error(y_va, model.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score:.4f}")

# スコアグラフ
df = pl.DataFrame({
    "Fold": [str(i) for i in range(folds)] * 2 + ["Final"],
    "RMSLE": list(train_scores) + list(valid_scores) + [test_score],
    "type": ["train"] * folds + ["valid"] * folds + ["test"],
})

plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="Fold", y="RMSLE", hue="type")

plt.title("RMSLE per Fold + Final Test Score")
plt.show()


In [ ]:
# Target vs Ridge predict error
ridge_err = y_train - model.predict(X_train)
err_df = pl.DataFrame({
    "SalePrice":y_train,
    "Error":ridge_err
})
sns.scatterplot(err_df,x='SalePrice',y='Error')


In [ ]:
sns.histplot(ridge_err)

### grid search(Ridge)

In [ ]:
alpha_candidates = [0.1, 1.0, 10.0, 100.0, 300.0]

# selectステップ(特徴量選択用Ridge)とridgeステップ(最終回帰用Ridge)のalphaを連動させて探索
param_grid = [
    {
        # "regressor__select__estimator__alpha": [alpha],
        "regressor__ridge__alpha": [alpha],
    }
    for alpha in alpha_candidates
]

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_log_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV RMSLE: {-grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
test_score_gs = root_mean_squared_log_error(y_va, best_model.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_gs:.4f}")

# alphaごとのCVスコア一覧
cv_results = pd.DataFrame(grid_search.cv_results_)[
    ["param_regressor__ridge__alpha", "mean_test_score", "std_test_score"]
].assign(
    mean_test_RMSLE=lambda df: -df["mean_test_score"]
).sort_values("param_regressor__ridge__alpha")
cv_results


In [ ]:
# 特徴量の寄与度（Ridge係数）トップ25
all_feature_names = model.regressor_.named_steps["prep"].get_feature_names_out()
# selected_mask = model.regressor_.named_steps["select"].get_support()
feature_names = all_feature_names  # [selected_mask]
coefs = model.regressor_.named_steps["ridge"].coef_

importance_df = pl.DataFrame({
    "feature": feature_names,
    "coef": coefs,
}).with_columns(
    pl.col("coef").abs().alias("abs_coef")
).sort("abs_coef", descending=True)

importance_df = importance_df.head(25)

plt.figure(figsize=(9, 10))
sns.barplot(
    data=importance_df.to_pandas(),
    x="coef", y="feature",
    hue="coef", palette="coolwarm", dodge=False, legend=False,
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top 25 of Ridge-coef")
plt.xlabel("coef (standardized)")
plt.ylabel("features")
plt.tight_layout()

plt.show()


### XGBoost

In [ ]:
def make_xgb(**xgb_params):
    base_model_xgb = Pipeline([
        ("prep", preprocessor),
        ("xgb", XGBRegressor(**xgb_params)),
    ])
    return TransformedTargetRegressor(
        regressor=base_model_xgb,
        transformer=log_standardize_y,
        check_inverse=False
    )

XGB_PARAMS = dict(
    n_estimators=2000,  # early stoppingが上限を決めるため大きめに設定
    learning_rate=0.01,
    max_leaves=15,
    max_depth=5,
    min_child_weight=50,
    reg_alpha=0.1,
    reg_lambda=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    importance_type='gain',
    early_stopping_rounds=50,
    eval_metric="rmse",
)

def fit_xgb_with_early_stopping(X_tr, y_tr, X_val, y_val, **xgb_params):
    """PipelineのprepとTransformedTargetRegressorの変換を手動で適用し、eval_setにearly stoppingを効かせる"""
    model = make_xgb(**xgb_params)
    prep = model.regressor.named_steps["prep"]
    xgb = model.regressor.named_steps["xgb"]

    y_tr_t = log_standardize_y.fit_transform(y_tr.to_numpy().reshape(-1, 1)).ravel()
    y_val_t = log_standardize_y.transform(y_val.to_numpy().reshape(-1, 1)).ravel()

    # prep内のTargetEncoderがyを要求するため、生のy_trを渡す
    X_tr_t = prep.fit_transform(X_tr, y_tr)
    X_val_t = prep.transform(X_val)

    xgb.fit(X_tr_t, y_tr_t, eval_set=[(X_val_t, y_val_t)], verbose=False)

    # 学習済みprep/xgbをそのままPipeline/TransformedTargetRegressorに差し込む
    model.regressor_ = model.regressor
    model.transformer_ = log_standardize_y
    model._training_dim = 1  # predict()内部で参照されるためfit()を通さない分手動設定
    return model

model_xgb = fit_xgb_with_early_stopping(X_train, y_train, X_va, y_va, **XGB_PARAMS)
best_iteration_xgb = model_xgb.regressor_.named_steps["xgb"].best_iteration
print(f"best_iteration: {best_iteration_xgb}")

test_score_xgb = root_mean_squared_log_error(y_va, model_xgb.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_xgb:.4f}")


### grid search (XGBoost)

In [ ]:
param_dist_xgb = {
    "regressor__xgb__n_estimators": [500, 800],
    "regressor__xgb__learning_rate": [0.02, 0.03, 0.05],
    "regressor__xgb__max_leaves": [5, 7],
    "regressor__xgb__max_depth": [3, 4],
    "regressor__xgb__min_child_weight": [30, 50, 70, 100],
    "regressor__xgb__reg_alpha": [0.01, 0.05, 0.1],
    "regressor__xgb__reg_lambda": [1.0, 2.0, 5.0, 10.0],
    "regressor__xgb__subsample": [0.5, 0.6, 0.7],
    "regressor__xgb__colsample_bytree": [0.5, 0.6, 0.7],
}

# early_stopping_rounds付きだとeval_setが必須になりCVと相性が悪いため、探索用は早期終了なしのモデルを使う
search_model_xgb = make_xgb(**{k: v for k, v in XGB_PARAMS.items() if k not in ("early_stopping_rounds", "eval_metric")})

random_search_xgb = RandomizedSearchCV(
    estimator=search_model_xgb,
    param_distributions=param_dist_xgb,
    scoring="neg_root_mean_squared_log_error",
    cv=cv,
    n_iter=40,
    n_jobs=-1,
    refit=True,
    random_state=42,
)
random_search_xgb.fit(X_train, y_train)

print(f"Best params: {random_search_xgb.best_params_}")
print(f"Best CV RMSLE: {-random_search_xgb.best_score_:.4f}")

# 探索で見つかったベストパラメータでearly stoppingを使い最終学習し直す
best_xgb_params = {
    k.replace("regressor__xgb__", ""): v
    for k, v in random_search_xgb.best_params_.items()
}
best_model_xgb = fit_xgb_with_early_stopping(
    X_train, y_train, X_va, y_va,
    **{**XGB_PARAMS, **best_xgb_params},
)
print(f"best_iteration: {best_model_xgb.regressor_.named_steps['xgb'].best_iteration}")

test_score_gs_xgb = root_mean_squared_log_error(y_va, best_model_xgb.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_gs_xgb:.4f}")

# パラメータごとのCVスコア一覧
cv_results_xgb = pd.DataFrame(random_search_xgb.cv_results_)[
    [
        "param_regressor__xgb__n_estimators",
        "param_regressor__xgb__learning_rate",
        "param_regressor__xgb__max_leaves",
        "param_regressor__xgb__max_depth",
        "param_regressor__xgb__min_child_weight",
        "param_regressor__xgb__reg_alpha",
        "param_regressor__xgb__reg_lambda",
        "mean_test_score",
        "std_test_score"
    ]
].assign(
    mean_test_RMSLE=lambda df: -df["mean_test_score"]

).sort_values("mean_test_RMSLE")
cv_results_xgb.sort_values('mean_test_RMSLE',ascending=True).head(15)

In [ ]:
# 特徴量の寄与度（XGBoost feature_importances_）トップ25
all_feature_names_xgb = best_model_xgb.regressor_.named_steps["prep"].get_feature_names_out()
# selected_mask_xgb = best_model_xgb.regressor_.named_steps["select"].get_support()
feature_names_xgb = all_feature_names_xgb  # [selected_mask_xgb]
importances_xgb = best_model_xgb.regressor_.named_steps["xgb"].feature_importances_

importance_df_xgb = pl.DataFrame({
    "feature": feature_names_xgb,
    "importance": importances_xgb,
}).sort("importance", descending=True)

importance_df_xgb = importance_df_xgb.head(25)

plt.figure(figsize=(9, 10))
sns.barplot(
    data=importance_df_xgb.to_pandas(),
    x="importance", y="feature",
    hue="importance", palette="coolwarm", dodge=False, legend=False,
)
plt.title("Top 25 of XGBoost feature_importances_")
plt.xlabel("importance")
plt.ylabel("features")
plt.tight_layout()

plt.show()


### SVR (kernel=rbf)

In [ ]:
base_model_svr = Pipeline([
    ("prep", preprocessor),
    ("svr", SVR(
        kernel="rbf",
        C=10,
        epsilon=0.05,
        gamma=0.001,
    ))
])
model_svr = TransformedTargetRegressor(
    regressor=base_model_svr,
    transformer=log_standardize_y,
    check_inverse=False
)

result_svr = cross_validate(
    model_svr, X_train, y_train,
    cv=cv,
    scoring="neg_root_mean_squared_log_error",
    return_train_score=True,
)

# スコア表示
train_scores_svr = -result_svr["train_score"]
valid_scores_svr = -result_svr["test_score"]

for i, (tr, va) in enumerate(zip(train_scores_svr, valid_scores_svr), 1):
    print(f"Fold {i}: train RMSLE = {tr:.4f} / valid RMSLE = {va:.4f}")
print(f"CV平均: train {train_scores_svr.mean():.4f} / valid {valid_scores_svr.mean():.4f}")

model_svr.fit(X_train, y_train)
test_score_svr = root_mean_squared_log_error(y_va, model_svr.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_svr:.4f}")

### grid search (SVR)

In [ ]:
param_dist_svr = {
    "regressor__svr__C": [0.5, 1.0, 3.0, 5.0, 10.0],
    "regressor__svr__epsilon": [0.01, 0.02, 0.05],
    "regressor__svr__gamma": [0.0005, 0.001, 0.005, 0.01],
}

random_search_svr = RandomizedSearchCV(
    estimator=model_svr,
    param_distributions=param_dist_svr,
    scoring="neg_root_mean_squared_log_error",
    cv=cv,
    n_iter=100,
    n_jobs=-1,
    refit=True,
    random_state=42,
)
random_search_svr.fit(X_train, y_train)

print(f"Best params: {random_search_svr.best_params_}")
print(f"Best CV RMSLE: {-random_search_svr.best_score_:.4f}")

best_model_svr = random_search_svr.best_estimator_
test_score_gs_svr = root_mean_squared_log_error(y_va, best_model_svr.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_gs_svr:.4f}")

# パラメータごとのCVスコア一覧
cv_results_svr = pd.DataFrame(random_search_svr.cv_results_)[
    [
        "param_regressor__svr__C",
        "param_regressor__svr__epsilon",
        "param_regressor__svr__gamma",
        "mean_test_score",
        "std_test_score"
    ]
].assign(
    mean_test_RMSLE=lambda df: -df["mean_test_score"]

).sort_values("mean_test_RMSLE")
cv_results_svr.sort_values('mean_test_RMSLE',ascending=True).head(15)

### stacking

In [ ]:
from sklearn.ensemble import StackingRegressor

# StackingRegressor/cross_validateは内部でクローンして再fitするため、
# early_stopping_rounds付きのbest_model_xgbはeval_setなしでエラーになる。
# best_iterationで木の本数を固定した早期終了なしのXGBoostを代わりに使う。
stacking_xgb_params = {
    k: v for k, v in {**XGB_PARAMS, **best_xgb_params}.items()
    if k not in ("early_stopping_rounds", "eval_metric")
}
stacking_xgb_params["n_estimators"] = best_model_xgb.regressor_.named_steps["xgb"].best_iteration + 1
stacking_xgb_estimator = make_xgb(**stacking_xgb_params)

stacking_model = StackingRegressor(
    estimators=[
        ("ridge", best_model),
        ("xgb", stacking_xgb_estimator),
        ("svr", best_model_svr),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=cv,
    n_jobs=-1,
    passthrough=False,
)

result_stack = cross_validate(
    stacking_model, X_train, y_train,
    cv=cv,
    scoring="neg_root_mean_squared_log_error",
    return_train_score=True,
)

# スコア表示
train_scores_stack = -result_stack["train_score"]
valid_scores_stack = -result_stack["test_score"]

for i, (tr, va) in enumerate(zip(train_scores_stack, valid_scores_stack), 1):
    print(f"Fold {i}: train RMSLE = {tr:.4f} / valid RMSLE = {va:.4f}")
print(f"CV平均: train {train_scores_stack.mean():.4f} / valid {valid_scores_stack.mean():.4f}")

stacking_model.fit(X_train, y_train)
test_score_stack = root_mean_squared_log_error(y_va, stacking_model.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score_stack:.4f}")

# スコアグラフ
df_stack = pl.DataFrame({
    "Fold": [str(i) for i in range(folds)] * 2 + ["Final"],
    "RMSLE": list(train_scores_stack) + list(valid_scores_stack) + [test_score_stack],
    "type": ["train"] * folds + ["valid"] * folds + ["test"],
})

plt.figure(figsize=(9, 5))
sns.barplot(data=df_stack, x="Fold", y="RMSLE", hue="type")

plt.title("Stacking: RMSLE per Fold + Final Test Score")
plt.show()

### submit

In [ ]:
from sklearn.base import clone

# ベストハイパーパラメータのRidge/XGBoost/SVRをベース推定器としたスタッキングモデルを
# 全学習データ(X, y)で最終学習
final_stacking_model = clone(stacking_model)
final_stacking_model.fit(X, y)

pred_final = final_stacking_model.predict(X_test)

submit_df = pl.DataFrame({
    "Id": test['Id'],
    "SalePrice": pred_final
})


In [ ]:
submit_df.write_csv("../submit.csv")

### submit (Ridgeのみ)

In [ ]:
from sklearn.base import clone

# Ridgeベストパラメータのみで全学習データ(X, y)を再学習
final_ridge_model = clone(best_model)
final_ridge_model.fit(X, y)

pred_final_ridge = final_ridge_model.predict(X_test)

submit_df_ridge = pl.DataFrame({
    "Id": test['Id'],
    "SalePrice": pred_final_ridge
})

submit_df_ridge.write_csv("../submit_ridge.csv")
